In [ ]:
import pandas as pd
import plotly.graph_objects as go

# PDF矢量导出需要安装 kaleido: pip install kaleido

# ============================================================
# 1. 读取数据
# ============================================================
df = pd.read_csv(r'D:\A.baumannii\data\sankey.csv')


# ============================================================
# 2. 标准化 ARG 名称
# ============================================================

# 先处理可能存在的不同形式的单引号/双引号
df["ARG"] = (
    df["ARG"]
    .astype(str)
    .str.strip()
    .str.replace("’", "'", regex=False)
    .str.replace("′", "'", regex=False)
    .str.replace("″", "''", regex=False)
)

# 将原来的基因名称替换为你要求的名称
arg_name_map = {
    "APH(3'')-Ib": "aph(3'')-Ib",
    "APH(6)-Id": "aph(6)-Id",

    "OXA-23": "blaOXA-23",
    "OXA-24": "blaOXA-24",
    "OXA-72": "blaOXA-72",

    "APH(3')-Ia": "aph(3')-Ia",
    "APH(3')-VIa": "aph(3')-VIa",
    "APH(3')-VI": "aph(3')-VI",

    "TEM-12": "blaTEM-12",
    "NDM-1": "blaNDM-1",

    "AAC(6')-Ib7": "aac(6')-Ib7",
    "AAC(3)-Ia": "aac(3)-Ia",
    "AAC(6')-Ian": "aac(6')-Ian",
    "AAC(6')-Iaf": "aac(6')-Iaf",
    "AAC(3)-IIe": "aac(3)-IIe",

    "ANT(2'')-Ia": "ant(2'')-Ia",

    "determinant_of_bleomycin_resistance": "bleO",

    "SAT-1": "sat-1"
}

df["ARG"] = df["ARG"].replace(arg_name_map)


# ============================================================
# 3. 指定耐药基因顺序
# ============================================================

arg_order = [
    "aph(3'')-Ib",
    "aph(6)-Id",
    "blaOXA-23",
    "tet(B)",
    "msrE",
    "mphE",
    "armA",
    "sul2",
    "sul1",
    "aph(3')-Ia",
    "blaTEM-12",
    "catB8",
    "aac(6')-Ib7",
    "aac(3)-Ia",
    "aph(3')-VIa",
    "ant(2'')-Ia",
    "blaOXA-24",
    "blaOXA-72",
    "bleO",
    "blaNDM-1",
    "aac(6')-Ian",
    "aadA2",
    "aac(6')-Iaf",
    "catI",
    "cmlB1",
    "aph(3')-VI",
    "cmlA5",
    "arr-2",
    "tet(39)",
    "aac(3)-IIe",
    "catB3",
    "floR",
    "aadA5",
    "tet(A)",
    "dfrA1",
    "dfrA19",
    "sat-1"
]

# 按你给出的顺序排列
# 只保留实际存在于 CSV 中的基因
present_args = set(df["ARG"])

args = [
    arg for arg in arg_order
    if arg in present_args
]

# 如果 CSV 中还有上述列表之外的 ARG，则放在最后，避免数据丢失
other_args = [
    arg for arg in df["ARG"].unique()
    if arg not in arg_order
]

args = args + other_args


# ============================================================
# 4. 获取 Drug_class 和 Mechanism
# ============================================================

drugs = df["Drug_class"].unique().tolist()
mechanisms = df["Mechanism"].unique().tolist()

all_nodes = args + drugs + mechanisms

node_indices = {
    node: i
    for i, node in enumerate(all_nodes)
}


# ============================================================
# 5. 构建 Sankey 图 Source、Target、Value
# ============================================================

source = []
target = []
value = []

# -------------------------
# 第一层：ARG -> Drug_class
# -------------------------
for _, row in df.iterrows():

    source.append(
        node_indices[row["ARG"]]
    )

    target.append(
        node_indices[row["Drug_class"]]
    )

    value.append(
        row["count"]
    )


# -------------------------
# 第二层：Drug_class -> Mechanism
# -------------------------
df_step2 = (
    df.groupby(
        ["Drug_class", "Mechanism"]
    )["count"]
    .sum()
    .reset_index()
)

for _, row in df_step2.iterrows():

    source.append(
        node_indices[row["Drug_class"]]
    )

    target.append(
        node_indices[row["Mechanism"]]
    )

    value.append(
        row["count"]
    )


# ============================================================
# 6. ARG 颜色
# ============================================================

arg_color_map = {

    # Aminoglycoside (yellow / olive tones)
    "aph(3'')-Ib": "#D8C56A",
    "aph(6)-Id": "#DCE8B8",
    "aph(3')-Ia": "#E6B37A",
    "aph(3')-VIa": "#C9B6D8",
    "aph(3')-VI": "#A9D18E",
    "aac(6')-Ib7": "#B7D7C2",
    "aac(3)-Ia": "#B9D7C6",
    "aac(6')-Ian": "#DCCB8A",
    "aac(6')-Iaf": "#C7D89A",
    "aac(3)-IIe": "#A7C58C",
    "ant(2'')-Ia": "#D5C4E8",
    "armA": "#8FC58A",
    "aadA2": "#C9DDA0",
    "aadA5": "#B8D49A",

    # Beta-lactam (blue-gray)
    "blaOXA-23": "#8FB4CF",
    "blaOXA-24": "#BFD7E6",
    "blaOXA-72": "#A7C9DE",
    "blaTEM-12": "#C7D8E5",
    "blaNDM-1": "#7EA9C8",

    # Phenicol (green-blue)
    "catB8": "#8FC8B5",
    "catB3": "#86BCA9",
    "catI": "#B6D9A8",
    "cmlB1": "#9DC8B2",
    "cmlA5": "#B2D7C4",
    "floR": "#7FB6B6",

    # Trimethoprim
    "dfrA1": "#9CC6C9",
    "dfrA19": "#A8D2C9",

    # Macrolide
    "msrE": "#A7D8C2",
    "mphE": "#8FC9D9",

    # Sulfonamide
    "sul2": "#C7A9D9",
    "sul1": "#E8C37A",

    # Bleomycin
    "bleO": "#B7D9B0",

    # Streptothricin
    "sat-1": "#A98BC7",

    # Tetracycline
    "tet(A)": "#E4D98A",
    "tet(B)": "#9CC4D5",
    "tet(39)": "#D8D59A",

    # Rifamycin
    "arr-2": "#E5B4C2"
}


# ============================================================
# 7. ARG 节点颜色
# ============================================================

arg_colors = [
    arg_color_map.get(arg, "#8EBAD9")
    for arg in args
]


# ============================================================
# 8. Drug_class 节点颜色
# ============================================================

drugs_color = "#E8DFAE"

drug_colors = [
    drugs_color
] * len(drugs)


# ============================================================
# 9. Mechanism 节点颜色
# ============================================================

mechanism_colors_list = [
    "#B8B8B8",
    "#8FBC8F",
    "#6FA8DC",
    "#76B7B2",
    "#D6A3D0"
]

mechanism_colors = [
    mechanism_colors_list[
        i % len(mechanism_colors_list)
    ]
    for i in range(len(mechanisms))
]


# 所有节点颜色
node_colors = (
    arg_colors
    + drug_colors
    + mechanism_colors
)


# ============================================================
# 10. 设置流线颜色
# ============================================================

link_colors = []


def hex_to_rgba(hex_color, alpha=0.35):
    """
    HEX 颜色转换为 rgba
    """
    hex_color = hex_color.lstrip("#")

    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)

    return f"rgba({r}, {g}, {b}, {alpha})"


# ------------------------------------------------------------
# 第一层：ARG -> Drug_class
# 流线颜色跟随 ARG
# ------------------------------------------------------------
for _, row in df.iterrows():

    arg_name = row["ARG"]

    arg_color = arg_color_map.get(
        arg_name,
        "#8EBAD9"
    )

    link_colors.append(
        hex_to_rgba(
            arg_color,
            0.30
        )
    )


# ------------------------------------------------------------
# 第二层：Drug_class -> Mechanism
# ------------------------------------------------------------
for _, row in df_step2.iterrows():

    link_colors.append(
        hex_to_rgba(
            drugs_color,
            0.30
        )
    )


# ============================================================
# ============================================================
# 11. ARG 名称格式化
# ============================================================

def format_arg_label(node):
    """
    ARG 科学命名格式：
    blaOXA-23 -> bla斜体，OXA-23正体下标
    aph(3'')-Ib 等 -> 全部斜体
    """

    if node.startswith("bla"):
        # bla 斜体；后续耐药基因编号下标且正体
        return (
            "<i>bla</i>"
            f"<sub>{node[3:]}</sub>"
        )

    else:
        return f"<i>{node}</i>"


plot_labels = []

for i, node in enumerate(all_nodes):

    if i < len(args):
        plot_labels.append(
            format_arg_label(node)
        )

    else:
        plot_labels.append(node)


# ============================================================
# 12. 绘制 Sankey
# ============================================================

fig = go.Figure(
    data=[
        go.Sankey(

            arrangement="snap",

            node=dict(
                pad=18,
                thickness=12,

                line=dict(
                    color="white",
                    width=0
                ),

                label=plot_labels,

                color=node_colors
            ),

            link=dict(
                source=source,
                target=target,
                value=value,
                color=link_colors
            )
        )
    ]
)


# ============================================================
# 13. 图形设置
# ============================================================

fig.update_layout(

    # #title_text="抗性基因-药物类别-作用机制",

    title_font=dict(
        family="Times New Roman",
        size=14
    ),

    font=dict(
        family="Times New Roman",
        size=12
    ),

    width=1500,
    height=900,
    paper_bgcolor="white",
    plot_bgcolor="white"
)


# ============================================================
# 14. 导出 PDF
# ============================================================

fig.write_image(
    "Dsankey_vector.png",
    width=1600,
    height=1000
)


# ============================================================
# 15. 显示
# ============================================================

# 不打开网页，仅生成PDF
# fig.show()


# ============================================================
# 16. 检查是否存在没有包含在目标列表中的 ARG
# ============================================================

if other_args:

    print("\n注意：CSV 中还存在以下未包含在指定列表中的 ARG：")

    for gene in other_args:
        print(gene)

else:
    print("\n所有 ARG 均已匹配指定的耐药基因名称。")



所有 ARG 均已匹配指定的耐药基因名称。
